# Ruff: Linting and Formatting

## The Core Question

**How do you keep your codebase clean and consistent?**

A 500-line simulation script written over six months, by two people, on two machines, will drift: unused imports pile up, spacing gets inconsistent, and a typo'd variable name hides until runtime. Ruff catches all of it in milliseconds.

## Why this library?

Ruff is a linter *and* formatter written in Rust — roughly **100x faster than flake8**. One tool replaces flake8, isort, and black. It finds **unused imports**, **undefined variables**, and **style inconsistencies**, and it can auto-fix most of them. For an engineer, that means: *consistent code = fewer bugs in long simulation scripts*, and a linter fast enough to run on every save.

## The Workflow

```
  write code (it gets messy)
        |
        v
  run:  ruff check file.py     -->  list the problems
        |
        v
  run:  ruff check --fix       -->  auto-fix what's safe
        |
        v
  run:  ruff format file.py    -->  normalize spacing/style
        |
        v
  clean, consistent, bug-free code
```


In [ ]:
import subprocess, sys
out = subprocess.run([sys.executable, '-m', 'ruff', '--version'],
                     capture_output=True, text=True)
print(out.stdout.strip())

## 1. A Messy Code Snippet

Here is a realistic "quick and dirty" heat-transfer script. It *works*, but look closely:

- `numpy`, `os`, and `sys` are imported but **never used**
- the import block is **unsorted**
- spacing is inconsistent (`h*A*(T_hot-T_cold)`, `def heat_transfer( h , A , ...)`)
- `undefined_name` is a **typo** that will crash at runtime
- `unused_variable` is dead code

This is exactly what accumulates in a real project.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

def heat_transfer( h , A , T_hot , T_cold ):
    # q = h * A * (T_hot - T_cold)
    q=h*A*(T_hot-T_cold)
    return q

def plot_result(x,y):
    plt.plot(x,y)
    plt.show()

unused_variable = 42
def broken():
    print(undefined_name)

## 2. Run `ruff check`

`ruff check` reports every lint problem with a **code** (e.g. `F401` = unused import, `F821` = undefined name) and the line.


In [ ]:
import subprocess, sys, tempfile, os

messy = '''import numpy as np
import matplotlib.pyplot as plt
import os
import sys

def heat_transfer( h , A , T_hot , T_cold ):
    # q = h * A * (T_hot - T_cold)
    q=h*A*(T_hot-T_cold)
    return q

def plot_result(x,y):
    plt.plot(x,y)
    plt.show()

unused_variable = 42
def broken():
    print(undefined_name)
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False, encoding='utf-8') as f:
    f.write(messy)
    path = f.name

result = subprocess.run([sys.executable, '-m', 'ruff', 'check', path],
                        capture_output=True, text=True)
print(result.stdout)
os.unlink(path)

### Reading the output

```
I001 [*] Import block is un-sorted or un-formatted
F401 [*] `numpy` imported but unused
F401 [*] `os` imported but unused
F401 [*] `sys` imported but unused
F821 Undefined name `undefined_name`
Found 5 errors.
[*] 4 fixable with `--fix`.
```

- **`F401`** (unused import) x3 — dead weight that misleads readers about what the script actually needs.
- **`I001`** (unsorted imports) — style drift.
- **`F821`** (undefined name) — the *real bug*. `undefined_name` will raise `NameError` the moment `broken()` runs. Ruff found it **without running the code**.

The `[*]` marks the four issues that `--fix` can resolve automatically.


## 3. Auto-Fix and Format

`ruff check --fix` applies the safe fixes (removing unused imports, sorting). `ruff format` then normalizes spacing and layout. Let's see the **before/after** diff.


In [ ]:
import subprocess, sys, tempfile, os

messy = '''import numpy as np
import matplotlib.pyplot as plt
import os
import sys

def heat_transfer( h , A , T_hot , T_cold ):
    # q = h * A * (T_hot - T_cold)
    q=h*A*(T_hot-T_cold)
    return q

def plot_result(x,y):
    plt.plot(x,y)
    plt.show()

unused_variable = 42
def broken():
    print(undefined_name)
'''

with tempfile.NamedTemporaryFile('w', suffix='.py', delete=False, encoding='utf-8') as f:
    f.write(messy)
    path = f.name

# Show what the formatter would change (dry run, no write)
result = subprocess.run([sys.executable, '-m', 'ruff', 'format', '--diff', path],
                        capture_output=True, text=True)
print('=== ruff format --diff ===')
print(result.stdout)
os.unlink(path)

### What changed?

The formatter:

- removed the unused `numpy`, `os`, `sys` imports (after `--fix`)
- sorted the import block
- normalized spacing: `heat_transfer(h, A, T_hot, T_cold)`, `q = h * A * (T_hot - T_cold)`
- added the two blank lines PEP 8 requires between top-level definitions

**Engineering takeaway:** none of this changes *what the code does* — it changes how *readable and maintainable* it is. In a 500-line CFD post-processing script, consistent formatting and no dead imports mean you can actually find the bug when it happens. And because Ruff is ~100x faster than flake8, you can run it on every save or every commit without waiting.


## 4. Exercises

1. **Fix it for real.** Save the messy snippet to `messy.py`. Run `ruff check --fix messy.py` then `ruff format messy.py`. Open the file and confirm the unused imports are gone and the spacing is clean.

2. **The undefined name.** `ruff check` flags `undefined_name` but `--fix` does *not* remove it (it can't guess your intent). What do you do? (Rename it, or delete the dead function.)

3. **Configure Ruff.** Create a `ruff.toml` that enables the `E`, `F`, and `I` rule sets and sets `line-length = 100`. Re-run `ruff check` on a file with a 120-character line. What new error appears?

4. **Run it on your project.** Point `ruff check` at a real simulation script of yours. How many issues does it find? Fix the auto-fixable ones and read the rest.
